# ScentAI — Offline Pipeline (One-Time)

This notebook consolidates all one-time offline processing for the ScentAI project.
It is designed to run on **Google Colab with an A100 (80 GB) or Blackwell 6000 GPU**.

## What this does
1. **LLM Enrichment** — fills all empty enrichment columns (`vibe_sentence`, `formality`, `day_night`, etc.)
   for all ~35k rows in `data/vibescent_enriched.csv` using Qwen3-8B (local GPU, no API keys).
2. **Corpus Re-embedding** — embeds every row's `retrieval_text` with `Qwen3-VL-Embedding-8B`
   and saves the resulting matrix to `artifacts/qwen3vl_corpus/embeddings.npy`.

## When to run
Run this notebook **once per dataset change**. After the artifacts are committed to the repo,
the live demo notebook (`harsh_week5_qwen3vl.ipynb`) loads them directly — this notebook does not
need to run again unless the CSV changes.

## Workflow: keeping Colab in sync with local changes

1. Edit code locally → run `./sync.sh` → pushes to GitHub
2. In Colab: **run the SYNC cell** (fast git pull, no reinstall) to pick up the latest code
3. On a fresh runtime: run **Cell 1 (Environment Setup)** once, restart, then continue from Stage 2

> **Note:** Cell 1 clones the repo from GitHub directly — no zip upload required.
> Google Drive is still mounted for the `uv` package cache and output artifacts.

## Stage Map
| Stage | Cell | Description |
|-------|------|-------------|
| SYNC | SYNC cell | Pull latest code from GitHub after `./sync.sh` — no reinstall |
| 1 | Cell 1 | Environment setup — git clone, install deps. **Restart runtime after this cell.** |
| 2 | Cell 2 | Config — paths, Drive mount for outputs, HF token |
| 3 | Cell 3 | Load & inspect dataset, resume from checkpoint if available |
| 4 | Cell 4 | LLM enrichment (Qwen3-8B, vLLM native guided decoding, batch 16) |
| 5 | Cell 5 | Corpus re-embedding (Qwen3-VL-Embedding-8B, batch 64) |
| 6 | Cell 6 | Verify artifacts |
| 7 | Cell 7 | Next steps — copy outputs to repo, update settings |


In [ ]:
# SYNC — Pull latest code from GitHub (run after ./sync.sh on your local machine)
# Pulls code only — no reinstall. Skip this cell on a brand-new runtime.
import subprocess, os

REPO_DIR = '/content/vibescent'

if os.path.exists(os.path.join(REPO_DIR, '.git')):
    result = subprocess.run(
        ['git', '-C', REPO_DIR, 'pull', 'origin', 'main'],
        capture_output=True, text=True,
    )
    print(result.stdout.strip() or 'Already up to date.')
    if result.returncode != 0:
        print('[WARN] git pull failed:', result.stderr.strip())
else:
    print('[OK] Repo not yet cloned — skip this cell on first run. Run Cell 1 (Environment Setup) first.')


In [ ]:
# Stage 1: Environment Setup
# Run once per new runtime, then restart runtime, then continue from Stage 2.
import subprocess, sys, traceback, os
import IPython

def custom_exc(shell, etype, evalue, tb, tb_offset=None):
    print("\n" + "="*60)
    print("!!! AN ERROR OCCURRED !!!")
    print("="*60)
    traceback.print_exception(etype, evalue, tb)
    print("="*60)
    print("!!! CHECK THE TRACEBACK ABOVE TO FIND THE EXACT LINE OF CODE !!!\n")

_ipython = IPython.get_ipython()
if _ipython:
    _ipython.set_custom_exc((Exception,), custom_exc)
    _ipython.magic("xmode Verbose")

REPO_DIR = '/content/vibescent'
GITHUB_REPO = 'https://github.com/GavinGongTech/VibeScent.git'

# Step 1: Clone or update repo from GitHub
# After ./sync.sh pushes to GitHub, use the SYNC cell (no reinstall needed).
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    print("Step 1: Repo already present — pulling latest from GitHub...")
    result = subprocess.run(
        ['git', '-C', REPO_DIR, 'pull', 'origin', 'main'],
        capture_output=True, text=True,
    )
    print(result.stdout.strip() or 'Already up to date.')
else:
    print("Step 1: Cloning from GitHub (first run)...")
    subprocess.check_call(['git', 'clone', '--depth=1', GITHUB_REPO, REPO_DIR])
    print(f"Cloned to {REPO_DIR}")

os.chdir(REPO_DIR)

# Step 2: Mount Drive for uv package cache + output artifacts
print("Step 2: Mounting Google Drive for package cache and outputs...")
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    UV_CACHE = '/content/drive/MyDrive/uv_cache'
    os.makedirs(UV_CACHE, exist_ok=True)
    os.environ['UV_CACHE_DIR'] = UV_CACHE
    print(f"uv cache: {UV_CACHE}")
except Exception as e:
    print(f"[WARN] Drive mount skipped ({e}). Package cache will be ephemeral.")

# Step 3: Install uv
print("Step 3: Installing uv...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'uv'], check=True)

# outlines removed — vLLM native guided decoding is used instead (no version conflict).
_pkgs = [
    'google-genai',
    'pandas',
    'numpy',
    'torch',
    'accelerate',
    'vllm',
    'bitsandbytes',
    'qwen-vl-utils>=0.0.14',
    'json-repair',
    'tqdm',
    'hf_transfer',
]
print("Step 4: Installing project and dependencies via uv...")
subprocess.run(['uv', 'pip', 'install', '--system', '-e', REPO_DIR] + _pkgs, check=True)

print('\nEnvironment ready. Restart runtime now, then continue from Stage 2.')


In [ ]:
# Stage 2: Config
import os, sys
REPO_DIR = '/content/vibescent'
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

# Google Drive — all outputs saved here (survives session restart)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_BASE = '/content/drive/MyDrive/vibescent_offline'
os.makedirs(DRIVE_BASE, exist_ok=True)

# Paths
INPUT_CSV      = os.path.join(REPO_DIR, 'data', 'vibescent_enriched.csv')
ENRICHED_CSV   = os.path.join(DRIVE_BASE, 'vibescent_enriched_full.csv')
CHECKPOINT_CSV = os.path.join(DRIVE_BASE, 'vibescent_enriched_full.csv.ckpt')
FAILURES_LOG   = os.path.join(DRIVE_BASE, 'enrichment_failures.jsonl')
EMBEDDINGS_DIR = os.path.join(DRIVE_BASE, 'qwen3vl_corpus')
os.makedirs(EMBEDDINGS_DIR, exist_ok=True)

print(f'Config ready. Drive base: {DRIVE_BASE}')
# Set Hugging Face cache to Google Drive to prevent redownloading models
HF_CACHE = '/content/drive/MyDrive/hf_cache'
import os
os.makedirs(HF_CACHE, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
try:
try:
    from google.colab import userdata as _ud
    hf_token = _ud.get('HF_TOKEN') or ''
except Exception:
    hf_token = os.environ.get('HF_TOKEN', '')
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
os.environ['HF_TOKEN'] = hf_token
if hf_token:
    print('HF_TOKEN loaded from Colab Secrets.')
else:
    print('[WARN] HF_TOKEN not set — add it via Colab left sidebar → Secrets.')
except Exception:
    pass


In [ ]:
# Stage 3: Load and Inspect Dataset
import pandas as pd, numpy as np

df_raw = pd.read_csv(INPUT_CSV)
print(f'Loaded: {df_raw.shape}')

ENRICHMENT_COLS = [
    'vibe_sentence', 'formality', 'day_night', 'fresh_warm',
    'likely_season', 'likely_occasion', 'character_tags',
    'mood_tags', 'color_palette', 'projection',
]
# Show how many rows actually need enrichment
for col in ENRICHMENT_COLS:
    filled = df_raw[col].notna().sum() if col in df_raw.columns else 0
    pct = filled / len(df_raw) * 100
    print(f'  {col}: {filled}/{len(df_raw)} filled ({pct:.1f}%)')

# Auto-resume: if checkpoint exists, use it as starting point
if os.path.exists(CHECKPOINT_CSV):
    df_ckpt = pd.read_csv(CHECKPOINT_CSV)
    print(f'\nCheckpoint found: {CHECKPOINT_CSV} ({df_ckpt.shape})')
    # Merge checkpoint into df_raw
    for col in ENRICHMENT_COLS:
        if col in df_ckpt.columns:
            df_raw[col] = df_raw.get(col, pd.Series(dtype='object'))
            mask = df_ckpt[col].notna()
            if mask.any() and col in df_raw.columns:
                df_raw.loc[mask[mask].index, col] = df_ckpt.loc[mask, col]
    already_done = df_raw['vibe_sentence'].notna().sum() if 'vibe_sentence' in df_raw.columns else 0
    print(f'After merge: {already_done} rows already enriched')

df_work = df_raw.copy()
if 'fragrance_id' not in df_work.columns:
    df_work.insert(0, 'fragrance_id', df_work.index.astype(str))
print(f'\nReady to enrich. Shape: {df_work.shape}')

In [ ]:
# Stage 4: LLM Enrichment (Qwen3-8B — local GPU, vLLM native guided decoding)
# Uses vLLM's built-in JSON-constrained decoding — no outlines dependency needed.
# Alternatives: model_name = "google/gemma-3-12b-it" or "google/gemma-3-27b-it"
import sys, os, json, traceback
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
import torch, gc
import pandas as pd
from tqdm.auto import tqdm

from vibescents.enrich import (
    VLLMNativeEnrichmentClient, build_retrieval_text,
    ENRICHMENT_COLUMNS, _build_prompt, _serialize_value
)

ENRICHMENT_MODEL = "Qwen/Qwen3-8B"  # change to any HF instruct model
CHECKPOINT_EVERY = 100
BATCH_SIZE = 16  # vLLM handles batching natively

gc.collect()
torch.cuda.empty_cache()

print(f"Loading enrichment model: {ENRICHMENT_MODEL}")
print(f"VRAM before: {torch.cuda.memory_allocated()/1e9:.1f} GB")

client = VLLMNativeEnrichmentClient(model_name=ENRICHMENT_MODEL)

print(f"VRAM after model load: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# Identify rows needing enrichment (skip already-filled rows on resume)
_need_mask = (
    df_work["vibe_sentence"].isna()
    if "vibe_sentence" in df_work.columns
    else pd.Series([True] * len(df_work), index=df_work.index)
)
_need_idx = df_work[_need_mask].index.tolist()
print(f"Rows needing enrichment: {len(_need_idx)} / {len(df_work)}")

for col in ENRICHMENT_COLUMNS:
    if col not in df_work.columns:
        df_work[col] = None

_completed, _failed = 0, 0
_failures = []

with tqdm(total=len(_need_idx), desc="Enriching (Batched)") as pbar:
    for i in range(0, len(_need_idx), BATCH_SIZE):
        batch_idx = _need_idx[i:i+BATCH_SIZE]
        batch_prompts = [_build_prompt(df_work.loc[idx]) for idx in batch_idx]

        try:
            records = client.generate_batch(batch_prompts)
            for idx, record in zip(batch_idx, records):
                if record is None:
                    _failed += 1
                    _failures.append({"idx": idx, "error": "Failed to parse vLLM output into EnrichmentSchemaV2."})
                else:
                    for col in ENRICHMENT_COLUMNS:
                        df_work.at[idx, col] = _serialize_value(getattr(record, col))
                    _completed += 1
        except Exception as e:
            error_msg = f"Batch failed: {str(e)}\n{traceback.format_exc()}"
            print(f"\n[ERROR] {error_msg}")
            for idx in batch_idx:
                _failed += 1
                _failures.append({"idx": idx, "error": error_msg})

        pbar.update(len(batch_idx))
        pbar.set_postfix(ok=_completed, fail=_failed)

        if (_completed + _failed) >= CHECKPOINT_EVERY and (_completed + _failed) % CHECKPOINT_EVERY < BATCH_SIZE:
            df_work.to_csv(CHECKPOINT_CSV, index=False)

# Final checkpoint + retrieval_text
df_work.to_csv(CHECKPOINT_CSV, index=False)
df_work = build_retrieval_text(df_work)
df_work.to_csv(ENRICHED_CSV, index=False)

if _failures:
    with open(FAILURES_LOG, "w") as fh:
        for r in _failures:
            fh.write(json.dumps(r) + "\n")

print(f"\nEnrichment done: {_completed} ok, {_failed} failed")
print(f"Saved: {ENRICHED_CSV}")
print("\nSample vibe_sentences:")
for _, row in df_work[df_work["vibe_sentence"].notna()].head(3).iterrows():
    print(f"  {row['name']}: {str(row['vibe_sentence'])[:100]}")


In [ ]:
# Stage 5: Corpus Re-embedding (Qwen3-VL-Embedding-8B)
# ONE-TIME: produces embeddings in the same vector space as query-time embedding
# Run AFTER Stage 4 so retrieval_text includes vibe_sentence + enriched fields
import time, torch, gc, json, glob
import numpy as np
import pandas as pd
from pathlib import Path
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

gc.collect(); torch.cuda.empty_cache()

# TF32 for faster matmuls on Blackwell/Ampere
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Load enriched CSV (from Stage 4 output or ENRICHED_CSV on Drive)
_csv = ENRICHED_CSV if os.path.exists(ENRICHED_CSV) else INPUT_CSV
df_embed = pd.read_csv(_csv)
print(f'Embedding {len(df_embed)} rows from: {_csv}')

# Build retrieval_text if missing
if 'retrieval_text' not in df_embed.columns or df_embed['retrieval_text'].isna().all():
    from vibescents.enrich import build_retrieval_text
    df_embed = build_retrieval_text(df_embed)

texts = df_embed['retrieval_text'].fillna(df_embed.get('name', '')).tolist()
print(f'Sample retrieval_text: {texts[0][:120]}')

# Load embedder — bump batch size for corpus throughput
from vibescents.embeddings import Qwen3VLMultimodalEmbedder
from vibescents.settings import Settings

Qwen3VLMultimodalEmbedder._BATCH_SIZE = 64  # throughput-optimised for corpus
_s = Settings(api_key=None)
embedder = Qwen3VLMultimodalEmbedder(settings=_s, load_in_8bit=False)

print(f'VRAM after embedder load: {torch.cuda.memory_allocated()/1e9:.1f} GB')

out_emb = os.path.join(EMBEDDINGS_DIR, 'embeddings.npy')
out_manifest = os.path.join(EMBEDDINGS_DIR, 'manifest.json')
CKPT_DIR = os.path.join(EMBEDDINGS_DIR, 'ckpts')
os.makedirs(CKPT_DIR, exist_ok=True)

# Resume logic
def get_checkpoints():
    files = sorted(glob.glob(os.path.join(CKPT_DIR, "embeddings_ckpt_*.npy")), 
                   key=lambda p: int(Path(p).stem.split("_")[-1]))
    return files

ckpt_files = get_checkpoints()
already_embedded = 0
for f in ckpt_files:
    already_embedded += np.load(f).shape[0]

print(f'Found {len(ckpt_files)} checkpoints. Already embedded: {already_embedded} / {len(texts)}')

texts_to_embed = texts[already_embedded:]

if texts_to_embed:
    print(f'Resuming embedding for remaining {len(texts_to_embed)} rows...')
    t0 = time.perf_counter()
    
    # We chunk the remaining texts so we can save periodic checkpoints
    CHUNK_SIZE = Qwen3VLMultimodalEmbedder._BATCH_SIZE * 50 # Checkpoint every 3200 rows
    
    from tqdm.auto import tqdm
    with tqdm(total=len(texts_to_embed), desc="Embedding (Outer Chunks)") as outer_pbar:
        for i in range(0, len(texts_to_embed), CHUNK_SIZE):
            chunk = texts_to_embed[i:i+CHUNK_SIZE]
            
            chunk_emb = embedder.embed_multimodal_documents(chunk)
            
            # Save checkpoint
            ckpt_idx = len(get_checkpoints())
            ckpt_path = os.path.join(CKPT_DIR, f"embeddings_ckpt_{ckpt_idx}.npy")
            np.save(ckpt_path, chunk_emb)
            
            outer_pbar.update(len(chunk))
        
    elapsed = time.perf_counter() - t0
    print(f'Embedded {len(texts_to_embed)} remaining rows in {elapsed:.0f}s  ({len(texts_to_embed)/elapsed:.0f} rows/s)')

# Load all checkpoints to form the final matrix
all_ckpts = get_checkpoints()
final_parts = [np.load(f) for f in all_ckpts]

if final_parts:
    embeddings = np.vstack(final_parts)
else:
    embeddings = np.empty((0, 0), dtype=np.float32)

print(f'Embeddings shape: {embeddings.shape}  dtype: {embeddings.dtype}')
assert embeddings.shape[0] == len(texts), f"Expected {len(texts)} embeddings, got {embeddings.shape[0]}"

# Verify L2 normalization
norms = np.linalg.norm(embeddings, axis=1)
print(f'Norms — mean: {norms.mean():.4f}  std: {norms.std():.5f}  min: {norms.min():.4f}')
print(f'NaN count: {np.isnan(embeddings).sum()}')
assert np.isnan(embeddings).sum() == 0, 'NaN values in embeddings!'

# Save
np.save(out_emb, embeddings)
manifest = {
    'model': 'Qwen/Qwen3-VL-Embedding-8B',
    'created': __import__('datetime').datetime.utcnow().isoformat(),
    'n_rows': int(embeddings.shape[0]),
    'dim': int(embeddings.shape[1]),
    'dtype': str(embeddings.dtype),
    'source_csv': _csv,
}
with open(out_manifest, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f'\nSaved embeddings: {out_emb}')
print(f'Saved manifest:   {out_manifest}')
print(json.dumps(manifest, indent=2))



In [ ]:
# Stage 6: Verify Artifacts
import numpy as np, json, os

# Load and spot-check embeddings
emb = np.load(os.path.join(EMBEDDINGS_DIR, 'embeddings.npy'))
with open(os.path.join(EMBEDDINGS_DIR, 'manifest.json')) as f:
    mf = json.load(f)

print(f'Embeddings shape: {emb.shape}')
print(f'NaN count: {np.isnan(emb).sum()}')
norms = np.linalg.norm(emb, axis=1)
print(f'Norms — mean: {norms.mean():.4f}  std: {norms.std():.5f}')
print(f'Manifest: {json.dumps(mf, indent=2)}')

# Sanity: top-5 most similar to row 0
sims = emb @ emb[0]
top5 = np.argsort(-sims)[1:6]
print(f'\nTop-5 most similar to "{df_embed.iloc[0]["name"]}":')
for i in top5:
    print(f'  [{i}] {df_embed.iloc[i]["name"]} — sim={sims[i]:.4f}')

print('\n=== All artifacts verified. Ready to commit to repo. ===')
print(f'Next step: copy {EMBEDDINGS_DIR}/embeddings.npy to artifacts/qwen3vl_corpus/ in the repo.')

# Stage 7: Next Steps — Copy Outputs to Repo

After all stages complete, copy the artifacts from Google Drive into the repo and commit:

```bash
# From repo root
mkdir -p artifacts/qwen3vl_corpus
cp /content/drive/MyDrive/vibescent_offline/qwen3vl_corpus/embeddings.npy artifacts/qwen3vl_corpus/
cp /content/drive/MyDrive/vibescent_offline/qwen3vl_corpus/manifest.json  artifacts/qwen3vl_corpus/
cp /content/drive/MyDrive/vibescent_offline/vibescent_enriched_full.csv   data/vibescent_enriched.csv
```

## Update `settings.py`

Change `corpus_embeddings_path` in `src/vibescents/settings.py` from the current value:

```python
# Before
corpus_embeddings_path: str = str(DEFAULT_ARTIFACTS_DIR / 'week2_precomputed' / 'fragrance_raw_full' / 'embeddings.npy')

# After
corpus_embeddings_path: str = str(DEFAULT_ARTIFACTS_DIR / 'qwen3vl_corpus' / 'embeddings.npy')
```

## Run the Week 5 Demo Notebook Next

Open `notebooks/harsh_week5_qwen3vl.ipynb`.
**Skip Stage 0** (corpus re-embedding) — it is now complete.
Continue from Stage 1 (setup) as normal.